In [ ]:
import pandas as pd
import hashlib
import os
import json
import re

models = [
    'gemini-2.0-flash',
    'claude-opus-4', 'claude-3.5-haiku',
    'llama-3.1-8b',  'llama-3.1-405b',
    'gpt-4o-mini', 'gpt-4o',
    'grok-3-mini', 'grok-3'
]

data_dir = "../data"
predictions_dir = "../predictions"
prompts_dir = "../prompts"
results_dir = "../results"

In [ ]:
question_map = {}
question_map_pro_con = {}

for file in os.listdir(data_dir):
    if not file.endswith('.json'):
        continue
    slug = file.replace('.json', '').lower()
    with open(os.path.join(data_dir, file), "r") as f:
        data = json.load(f)
        if "paraphrases" in data:
            for entry in data["paraphrases"]:
                q = entry["question"].strip().lower()
                pro_con = entry.get("orientation", "").strip().lower()
                question_map[q] = slug
                question_map_pro_con[q] = pro_con

## helper functions
def map_to_question(request_text):
    question = request_text.split('\n')[0].split('|')[0]
    return question_map[question]

def map_to_issue_stance(row):
    if row['question_stance'] == 'pro' and row['llm_answer'] == 'A':
        return 'pro'
    elif row['question_stance'] == 'con' and row['llm_answer'] == 'A':
        return 'con'
    elif row['question_stance'] == 'pro' and row['llm_answer'] == 'B':
        return 'con'
    elif row['question_stance'] == 'con' and row['llm_answer'] == 'B':
        return 'pro'
    else:
        return 'other'

answer_regexes = []

answer_regexes += [
    re.compile(r'position ([A|B])', re.IGNORECASE),
    re.compile(r'position <<([A|B])>>', re.IGNORECASE),
    re.compile(r"<<([A|B])>>", re.IGNORECASE),
    re.compile(r"^\s*([A|B])\s*$", re.IGNORECASE),
]

def extract_answer(text):
    for regex in answer_regexes:
        match = re.search(regex, text)
        if match:
            return match.group(1)
    return 'Other'

In [ ]:

split_regexes = [
    re.compile(r"Carefully weigh all presented evidence and answer either 'position <<A>>' or 'position <<B>>'.", re.IGNORECASE),
    re.compile(r"Carefully weigh all presented evidence and answer either 'position <<B>>' or 'position <<A>>'.", re.IGNORECASE),
    re.compile(r"Answer either 'position <<A>>' or 'position <<B>>' based on provided information.", re.IGNORECASE),
    re.compile(r"Answer either 'position <<B>>' or 'position <<A>>' based on provided information.", re.IGNORECASE),
]

def extract_evidence(text):
    output = []
    # try:
    if text is not None:
        for regex in split_regexes:
            match = regex.search(text)
            if match:
                text = text[match.end():]
                evidence = [t.strip() for t in text.split('---')][:-1]
                for e in evidence:
                    # print(e)
                    try:
                        text, citations = e.split('\nCitations:\n', 1)
                        # print(len(citations.split('\n')))
                    except:
                        text = e
                        citations = ''
                    out_cites = []
                    cites = set(re.findall(r'(\[\d+\])', text))
                    # print(cites)
                    # print('initial cite number', len(set(citations.split('\n'))))
                    for line in citations.split('\n'):
                        if any(line.startswith(c) for c in cites):
                            out_cites.append(line)
                    # print('final cite number', len(out_cites))
                    if citations:
                        out = "\n".join(set(out_cites))
                        e = f'{text}\nCitations:\n{out}'
                    else:
                        e = text
                    output.append(e.strip())
    return output
    # except:
    #     print(text)
    #     # print('no evidence found')
    #     return None



In [ ]:
def evidence_effect(pred_df, prompt_df, evidence_hashes):
    joined_df = pd.merge(pred_df, prompt_df, on='custom_id_hash', how='left')

    # First, explode evidence_hashes so each row is one evidence per answer
    exploded = joined_df.explode('evidence_hashes')

    # Get the baseline pro proportion for each (issue, question) with no evidence
    no_evidence = exploded[exploded['evidence_hashes'].isna()]
    pro_counts = no_evidence.groupby(['issue', 'question'])['issue_stance'].apply(lambda x: (x == 'pro').sum())
    pro_df = pro_counts.reset_index(name='pro_count')
    pro_df['proportion'] = pro_df['pro_count'] / 15

    # For each (issue, question, evidence_hash), calculate the pro count and total count with that evidence
    with_evidence = exploded[exploded['evidence_hashes'].notna()]
    # with_evidence = with_evidence[~with_evidence['evidence_case'].isin(['pro', 'con'])]
    pro_counts_evidence = with_evidence.groupby(['issue', 'question', 'evidence_hashes'])['issue_stance'].apply(lambda x: (x == 'pro').sum())
    total_counts_evidence = with_evidence.groupby(['issue', 'question', 'evidence_hashes'])['issue_stance'].count()
    pro_df_evidence = pro_counts_evidence.reset_index(name='pro_count_with_evidence')
    pro_df_evidence['total_with_evidence'] = total_counts_evidence.values
    pro_df_evidence['proportion_with_evidence'] = pro_df_evidence['pro_count_with_evidence'] / pro_df_evidence['total_with_evidence']

    # Merge baseline and with-evidence proportions
    merged = pd.merge(
        pro_df_evidence,
        pro_df,
        on=['issue', 'question'],
        how='left'
    )

    # Calculate the shift in pro proportion for each (issue, question, evidence_hash)
    merged['proportion_shift'] = merged['proportion_with_evidence'] - merged['proportion']

    # For each evidence_hash, calculate the average shift in pro proportion across all (issue, question) pairs where it appears,
    # weighted by the number of times that evidence appears for each (issue, question)
    def weighted_mean(df):
        return (df['proportion_shift'] * df['total_with_evidence']).sum() / df['total_with_evidence'].sum()

    average_shift_per_evidence = merged.groupby('evidence_hashes').apply(weighted_mean).reset_index(name='average_proportion_shift')
    average_shift_per_evidence['evidence_text'] = average_shift_per_evidence['evidence_hashes'].map(evidence_hashes)
    return average_shift_per_evidence

In [ ]:
evidence_models = {}

for model in models:
    evidence_hashes = {}
    
    print(f"Processing {model}...")

    print("  loading prompt data...")
    prompt_df = pd.read_json(os.path.join(prompts_dir, f"prompts_{model}.jsonl"), lines=True)

    if 'gpt' in model:
        print("  loading predictions...")
        pred_df = pd.read_json(os.path.join(predictions_dir, f"predictions_{model}.jsonl"), lines=True)

        print('  extracting prompt...')
        prompt_df['prompt'] = prompt_df['body'].apply(lambda x: x['messages'][0]['content'])

        print('  extracting response text...')
        pred_df['response_text'] = pred_df['response'].apply(lambda x: x['body']['choices'][0]['message']['content'].strip() if 'body' in x and len(x['body']['choices']) > 0 and 'message' in x['body']['choices'][0] and 'content' in x['body']['choices'][0]['message'] else None)

        print('  extracting custom id hash...')
        prompt_df['custom_id_hash'] = prompt_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        pred_df['custom_id_hash'] = pred_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())

    elif 'llama' in model:
        print("  loading predictions...")
        pred_df = pd.concat([pd.read_json(os.path.join(predictions_dir, f"predictions_{model}_{i}.jsonl"), lines=True) for i in range(3)])

        print('  extracting prompt...')
        prompt_df['prompt'] = prompt_df['body'].apply(lambda x: x['messages'][0]['content'])

        print('  extracting response text...')
        pred_df['response_text'] = pred_df['response'].apply(lambda x: x['choices'][0]['message']['content'].strip() if 'choices' in x and len(x['choices']) > 0 and 'message' in x['choices'][0] and 'content' in x['choices'][0]['message'] else None)

        print('  extracting custom id hash...')
        prompt_df['custom_id_hash'] = prompt_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        pred_df['custom_id_hash'] = pred_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())

    elif 'grok' in model:
        print("  loading predictions...")
        pred_df = pd.read_json(os.path.join(predictions_dir, f"predictions_{model}.jsonl"), lines=True)

        print('  extracting response text...')
        pred_df['response_text'] = pred_df['response']

        print('  extracting custom id hash...')
        prompt_df['custom_id_hash'] = prompt_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        pred_df['custom_id_hash'] = pred_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        
    elif 'claude' in model:
        print("  loading predictions...")
        if 'opus' in model:
            pred_df = pd.concat([pd.read_json(os.path.join(predictions_dir, f"predictions_{model}_{i}.jsonl"), lines=True) for i in range(2)])
        else:
            pred_df = pd.read_json(os.path.join(predictions_dir, f"predictions_{model}.jsonl"), lines=True)

        print('  extracting prompt...')
        prompt_df['prompt'] = prompt_df['request'].apply(lambda x: x['messages'][0]['content'])

        print('  extracting response text...')
        pred_df['response_text'] = pred_df['response'].apply(lambda x: x['content'][0]['text'] if 'content' in x and len(x['content']) > 0 and 'text' in x['content'][0] else None)

        print('  extracting custom id hash...')
        prompt_df['custom_id_hash'] = prompt_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        pred_df['custom_id_hash'] = pred_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())

    elif 'gemini' in model:
        print("  loading predictions...")
        pred_df = pd.read_json(os.path.join(predictions_dir, f"predictions_{model}.jsonl"), lines=True)

        print('  extracting prompt...')
        prompt_df['prompt'] = prompt_df['request'].apply(lambda x: x['contents'][0]['parts'][0]['text'])

        print('  extracting response text...')
        pred_df['response_text'] = pred_df['request'].apply(lambda x: x['contents'][0]['parts'][0]['text'].strip())

        print('  extracting custom id...')
        prompt_df['custom_id'] = prompt_df['request'].apply(lambda x: x['labels']['custom_id'])
        pred_df['custom_id'] = pred_df['request'].apply(lambda x: x['labels']['custom_id'])
        prompt_df['custom_id_hash'] = prompt_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
        pred_df['custom_id_hash'] = pred_df['custom_id'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
    
    print('  extracting evidence...')
    prompt_df['evidence'] = prompt_df['prompt'].apply(extract_evidence)

    print('  extracting evidence hashes...')
    prompt_df['evidence_hashes'] = prompt_df['evidence'].apply(
        lambda evidences: [hashlib.sha256(e.encode()).hexdigest() for e in evidences] if evidences is not None else None
    )
    
    for i, row in prompt_df.iterrows():
        if row['evidence_hashes'] is not None:
            for i, evidence_hash in enumerate(row['evidence_hashes']):
                if evidence_hash not in evidence_hashes:
                    evidence_hashes[evidence_hash] = row['evidence'][i]

    print("  extracting answers...")
    pred_df = pred_df[pred_df['response_text'].notna()]
    pred_df['llm_answer'] = pred_df['response_text'].apply(extract_answer)

    print("  processing answers...")
    pred_df['question'] = pred_df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
    pred_df['question_stance'] = pred_df['question'].map(question_map_pro_con)
    pred_df['issue'] = pred_df['question'].map(question_map)
    pred_df['evidence_case'] = pred_df['custom_id'].str.split('-evidence-').str[1]
    pred_df['issue_stance'] = pred_df.apply(map_to_issue_stance, axis=1)
    
    print("  computing evidence effect...")
    evidence_models[model] = evidence_effect(pred_df, prompt_df, evidence_hashes)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np

model_names = list(evidence_models.keys())
n_models = len(model_names)

fig, axes = plt.subplots(n_models, n_models, figsize=(4 * n_models, 4 * n_models), squeeze=False)

for i in range(n_models):
    for j in range(n_models):
        ax = axes[i, j]
        # Only plot on the lower triangle (i > j)
        if i > j:
            model_a = model_names[j]
            model_b = model_names[i]
            df_a = evidence_models[model_a][['evidence_hashes', 'average_proportion_shift']]
            df_b = evidence_models[model_b][['evidence_hashes', 'average_proportion_shift']]
            merged = df_a.merge(df_b, on='evidence_hashes', suffixes=(f'_{model_a}', f'_{model_b}'))
            x = merged[f'average_proportion_shift_{model_a}']
            y = merged[f'average_proportion_shift_{model_b}']
            if len(x) > 1:
                corr, pval = pearsonr(x, y)
            else:
                corr, pval = np.nan, np.nan
            ax.scatter(x, y, alpha=0.5)
            ax.plot([-1, 1], [-1, 1], 'k--', lw=1)
            ax.set_xlim(-1, 1)
            ax.set_ylim(-1, 1)
            ax.grid(True, linestyle='--', alpha=0.3)
            ax.set_title(f'r={corr:.2f}, p={pval:.5g}')
            # if not np.isnan(corr) and corr > 0.8:
            #     for spine in ax.spines.values():
            #         spine.set_edgecolor('red')
            #         spine.set_linewidth(3)
        else:
            ax.axis('off')

# Set model names on the left (y-axis) and bottom (x-axis)
for i, model in enumerate(model_names):
    axes[i,0].set_ylabel(f'{model}', fontsize=14)
    axes[-1,i].set_xlabel(f'{model}', fontsize=14)

# Add a main title to the figure
# fig.suptitle('Correlation of Evidence Effects Across Models', fontsize=24, y=1.02)

plt.tight_layout()
plt.savefig('../figures/evidence_correlation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import matplotlib.patches as patches
import matplotlib as mpl

model_names = list(evidence_models.keys())
n_models = len(model_names)

# Create figure with tighter bounds around the lower triangle
fig, axes = plt.subplots(n_models, n_models, figsize=(3.5 * n_models, 3.5 * n_models), squeeze=False)

# Function to check if two models are from the same family
def same_family(model_a, model_b):
    prefixes = ['llama', 'grok', 'gpt', 'claude', 'gemini']
    for prefix in prefixes:
        if model_a.startswith(prefix) and model_b.startswith(prefix):
            return True
    return False

# Store correlation values for heatmap
correlation_matrix = np.full((n_models, n_models), np.nan)

for i in range(n_models):
    for j in range(n_models):
        ax = axes[i, j]
        # Only plot on the lower triangle (i > j)
        if i > j:
            model_a = model_names[j]
            model_b = model_names[i]
            df_a = evidence_models[model_a][['evidence_hashes', 'average_proportion_shift']]
            df_b = evidence_models[model_b][['evidence_hashes', 'average_proportion_shift']]
            merged = df_a.merge(df_b, on='evidence_hashes', suffixes=(f'_{model_a}', f'_{model_b}'))
            x = merged[f'average_proportion_shift_{model_a}']
            y = merged[f'average_proportion_shift_{model_b}']
            if len(x) > 1:
                corr, pval = pearsonr(x, y)
            else:
                corr, pval = np.nan, np.nan
            
            # Store correlation for heatmap
            correlation_matrix[i, j] = corr
            
            # Check if models are from same family
            is_same_family = same_family(model_a, model_b)
            # Check if correlation is high
            is_high_corr = not np.isnan(corr) and corr > 0.7
            
            # Set background color based on correlation (heatmap effect)
            if not np.isnan(corr):
                # Normalize correlation to 0-1 range for color intensity
                # Use absolute value so negative correlations also show up
                color_intensity = min(abs(corr), 1.0)
                ax.set_facecolor(plt.cm.Blues(color_intensity))
            else:
                ax.set_facecolor('white')
            
            ax.scatter(x, y, alpha=0.7, color='black', s=20)
            ax.plot([-1, 1], [-1, 1], 'k--', lw=1)
            ax.set_xlim(-1, 1)
            ax.set_ylim(-1, 1)
            ax.grid(True, linestyle='--', alpha=0.3)

            if is_same_family:
                # Same family only - use blue border
                for spine in ax.spines.values():
                    spine.set_edgecolor('black')
                    spine.set_linewidth(5)
            
            # Add border highlighting
            # if is_same_family and is_high_corr:
            #     # Both same family and high correlation - use thick blue border
            #     for spine in ax.spines.values():
            #         spine.set_edgecolor('blue')
            #         spine.set_linewidth(4)
            # elif is_same_family:
            #     # Same family only - use blue border
            #     for spine in ax.spines.values():
            #         spine.set_edgecolor('blue')
            #         spine.set_linewidth(3)
            # elif is_high_corr:
            #     # High correlation only - use green border
            #     for spine in ax.spines.values():
            #         spine.set_edgecolor('green')
            #         spine.set_linewidth(3)
            
            ax.set_title(f'r={corr:.2f}, p={pval:.5g}', fontsize=12)
        else:
            ax.axis('off')

# Set model names on the left (y-axis) and bottom (x-axis)
for i, model in enumerate(model_names):
    axes[i,0].set_ylabel(f'{model}', fontsize=16)
    axes[-1,i].set_xlabel(f'{model}', fontsize=16)

fig.suptitle('Correlation of Argument Effects Across Models', 
             fontsize=40, y=0.82)


# Add a colorbar for correlation (r) intensity
norm = mpl.colors.Normalize(vmin=0, vmax=1)
sm = mpl.cm.ScalarMappable(cmap=plt.cm.Blues, norm=norm)
sm.set_array([])  # Only needed for older matplotlib

# Add a legend for the border meaning
legend_patch = patches.Patch(edgecolor='black', linewidth=5, label='Same model family')

# Place the legend for the border
fig.legend(handles=[legend_patch], loc='upper right', bbox_to_anchor=(0.8, 0.72), fontsize=24)

# Add a colorbar for the correlation intensity
# Define a new axes for the colorbar above the grid
cbar_ax = fig.add_axes([0.25, 0.76, 0.5, 0.02])  # [left, bottom, width, height] in figure fraction
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Correlation (|r|)', fontsize=20)
cbar.ax.tick_params(labelsize=16)

# Adjust layout to be tighter around the lower triangle
# plt.tight_layout(rect=[0, 0, 1, 1])

plt.savefig('../figures/evidence_correlation_heatmap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# After the correlation_matrix is filled in the plotting loop, calculate medians:
same_family_corrs = []
diff_family_corrs = []
for i in range(n_models):
    for j in range(n_models):
        if i > j and not np.isnan(correlation_matrix[i, j]):
            model_a = model_names[j]
            model_b = model_names[i]
            if same_family(model_a, model_b):
                same_family_corrs.append(correlation_matrix[i, j])
            else:
                diff_family_corrs.append(correlation_matrix[i, j])

median_same_family = np.median(same_family_corrs) if same_family_corrs else np.nan
median_diff_family = np.median(diff_family_corrs) if diff_family_corrs else np.nan

print(f"Median correlation (same family): {median_same_family:.3f}")
print(f"Median correlation (different family): {median_diff_family:.3f}")